# 05 • ค้นหา กฏ ด้วย Semantic Search

เป้าหมาย: สร้าง Embedding ของข้อความงานรับซื้อ QC คลังสินค้า และขนส่ง แล้วจัดอันดับด้วย cosine similarity ยังไม่ให้ LLM สร้างคำตอบ

**ข้อมูลสมมติสำหรับการอบรม:** ชื่อหน่วยงาน Lot ผู้ขาย ค่าที่วัด และ กฏ ในตัวอย่างถูกสร้างขึ้นเพื่อเรียนรู้ AI ไม่ใช่ข้อมูลการผลิตหรือระเบียบจริงของกลุ่มศรีตรัง

[![Open in Colab](https://img.shields.io/badge/Open_in_Colab-F9AB00?style=flat-square&logo=googlecolab&logoColor=white)](https://colab.research.google.com/drive/1D7zy4-lcLgiwqzG0FP6djdctFH_6MCrX?usp=sharing)


## เตรียมพร้อมก่อนเริ่ม
แทนที่ `your_api_key` ในตัวแปร `OPENAI_API_KEY` ด้วย API key จริงของคุณ แล้วรัน cells จากบนลงล่าง ใช้รูปแบบเดียวกันทั้ง local และ Colab โดย cell ติดตั้ง packages อยู่ใน notebook แล้ว

ก่อนแชร์หรือ commit notebook ให้เปลี่ยน API key กลับเป็น `your_api_key`


In [ ]:
%pip install -q openai numpy

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = "your_api_key"  # แทนที่ด้วย API key จริงของคุณ
client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4.1-mini"


## ข้อความที่แบ่งเป็น chunks แล้ว

การแบ่งเอกสารเป็น chunks ทำได้หลายวิธี เช่น แบ่งตามจำนวนตัวอักษร ย่อหน้า หรือหัวข้อ รวมถึงให้ LLM อ่านเอกสารและใช้ tools ช่วยแบ่งข้อความตามเนื้อหา

เพื่อให้ notebook นี้เรียบง่าย เราสมมติว่าเอกสารถูกแบ่งเป็น chunks ในรายการด้านล่างแล้ว และเริ่มจากสร้าง embeddings เพื่อค้นหาข้อความที่เกี่ยวข้อง


In [ ]:
chunks = [
    "[กฏ-1 รับน้ำยาง] บันทึก Lot ผู้ขาย น้ำหนักสุทธิ ทะเบียนรถ และเวลารับเข้า เก็บรหัสผู้ขายเพื่อเชื่อมโยงข้อมูล Traceability",
    "[กฏ-2 QC] เก็บผล DRC พร้อมรหัสตัวอย่างและ Lot หากผลยังไม่ครบ ให้ระบุสถานะรอ QC ห้ามสรุปว่าผ่านเกณฑ์จากข้อมูลที่หายไป",
    "[กฏ-3 คลังสินค้า] Lot ที่รอ QC ต้องแยกในพื้นที่พักรอผลและติดป้าย Lot ตรวจสถานะก่อนยืนยันการจัดส่ง",
    "[กฏ-4 ประสานงาน] ในสถานการณ์ฝึกอบรม ระดับน้ำตั้งแต่ 30 ซม. ให้ประสานหัวหน้าคลังและฝ่ายขนส่งเพื่อตรวจสอบแผนรับส่งสินค้า",
    "[กฏ-5 น้ำท่วม] ในสถานการณ์ฝึกอบรม ระดับน้ำตั้งแต่ 50 ซม. ให้พักแผนโหลดสินค้าของพื้นที่ที่ได้รับผลกระทบ แจ้งหัวหน้าคลัง ฝ่ายขนส่ง และ EHS เพื่อประเมินตามแผนของพื้นที่",
    "[กฏ-6 ส่งมอบ] เมื่อเวลารับส่งเปลี่ยน ให้ฝ่ายขนส่งประสานฝ่ายขาย แจ้งกำหนดการใหม่ที่ยืนยันแล้ว ห้ามรับรองเวลาส่งมอบที่ยังไม่ยืนยัน",
    "[กฏ-7 ซ่อมบำรุง] เมื่อพบปัญหาเครื่องจักร ให้บันทึกอาการ รหัสเครื่อง และเวลา แล้วส่งใบงานให้ผู้รับผิดชอบซ่อมบำรุง",
    "[กฏ-8 สิ่งแวดล้อม] หากพบข้อร้องเรียนด้านกลิ่นหรือน้ำเสีย ให้บันทึกเวลา จุดที่พบ และรายละเอียด ส่งต่อผู้รับผิดชอบสิ่งแวดล้อมเพื่อตรวจสอบ"
]

In [ ]:
import numpy as np

EMBED_MODEL = "text-embedding-3-small"

def embed(texts):
    result = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([item.embedding for item in result.data])

vectors = embed(chunks)
print("ขนาด Embedding matrix:", vectors.shape)

In [ ]:
def retrieve(question, k=2):
    query = embed([question])[0]
    scores = vectors @ query / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(query))
    indices = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in indices]

In [ ]:
question = "Lot ที่ยังไม่มีผล QC ควรจัดเก็บและตรวจสอบอย่างไร?"
for text, score in retrieve(question, k=3):
    print(round(score, 3), text)

## ลองทำด้วยตนเอง
1. ถามเรื่องการพัก Lot ที่ยังไม่มีผล QC ด้วยคำอื่น
2. เปลี่ยน k จาก 1 เป็น 3 แล้วเปรียบเทียบข้อความที่พบ
3. ถามเรื่องที่ไม่มีใน กฏ เช่น สวัสดิการพนักงาน อันดับสูงสุดแปลว่ามีคำตอบจริงหรือไม่?
